# Optimización de Hiperparámetros (Random Forest)

En las fases anteriores comprobamos que el dataset original en inglés arrojó mejores métricas de clasificación que el dataset traducido mediante Hugging Face. Por tanto, para esta fase de ajuste de hiperparámetros (*fine-tuning*), trabajaremos exclusivamente sobre el conjunto de datos nativo en inglés.

Para optimizar el uso de los recursos computacionales y evitar tiempos de procesamiento inasumibles, dividiremos la optimización en dos enfoques iterativos:
*   **Búsqueda Aleatoria (`RandomizedSearchCV`):** Exploración inicial de un espacio amplio de hiperparámetros mediante muestreo para acotar las zonas de mayor rendimiento.
*   **Búsqueda Exhaustiva (`GridSearchCV`):** Afinamiento localizado sobre los rangos detectados en la fase anterior (se abordará más adelante).

En la siguiente celda se reconstruye el pipeline de procesamiento de texto (NLP con spaCy, vectorización TF-IDF con ngramas 1-2) y se aplica el balanceo de clases sintético mediante SMOTE, generando las matrices definitivas de entrenamiento y test.

In [1]:
import pandas as pd
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

# Carga del dataset base sin traducciones (Capa Silver)
df = pd.read_parquet("../data/processed/df_final_silver.parquet")

# Concatenación de variables categóricas para formar la variable objetivo (Tripleta)
df['target_tripleta'] = df['queue'] + " - " + df['type'] + " - " + df['priority']

# Fusión del asunto y el cuerpo del mensaje para conformar el corpus principal
df['full_text'] = (df['subject'] + " " + df['body']).str.strip()

# Se eliminan clases con frecuencia menor a 2 para evitar errores matemáticos durante la partición estratificada
conteo = df['target_tripleta'].value_counts()
clases_validas = conteo[conteo > 1].index
df = df[df['target_tripleta'].isin(clases_validas)]

# Partición del dataset (80/20) manteniendo la proporción de las clases subyacentes
X = df['full_text']
y = df['target_tripleta']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# Carga del modelo lingüístico en inglés
nlp_en = spacy.load('en_core_web_sm')

# Función de normalización: aplica lematización y omite palabras vacías (stop words), signos de puntuación y números
def limpiar_texto_en(texto):
    doc = nlp_en(texto.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.like_num]
    return " ".join(tokens)

print("Procesando corpus de texto mediante spaCy (Lematización y eliminación de ruido sintáctico)...")
X_train_limpio = X_train.apply(limpiar_texto_en)
X_test_limpio = X_test.apply(limpiar_texto_en)

# Vectorización de los textos mediante TF-IDF. 
# Se incluyen bigramas (ngram_range=(1,2)) para capturar contexto técnico en pares de palabras.
print("Aplicando vectorización TF-IDF (Límite dimensional: 10.000 características)...")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train_limpio)
X_test_tfidf = tfidf.transform(X_test_limpio)

# Codificación numérica de la variable objetivo categórica (Label Encoding)
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Balanceo geométrico del conjunto de entrenamiento mediante generación de observaciones sintéticas
print("Aplicando técnica SMOTE al conjunto de entrenamiento...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote_encoded = smote.fit_resample(X_train_tfidf, y_train_encoded)

print(f"Dimensiones de la matriz de entrenamiento tras el balanceo: {X_train_smote.shape}")

Procesando corpus de texto mediante spaCy (Lematización y eliminación de ruido sintáctico)...
Aplicando vectorización TF-IDF (Límite dimensional: 10.000 características)...
Aplicando técnica SMOTE al conjunto de entrenamiento...
Dimensiones de la matriz de entrenamiento tras el balanceo: (159348, 10000)


### Búsqueda Aleatoria de Hiperparámetros

Definimos un diccionario con diferentes valores para los parámetros estructurales del modelo Random Forest (número de estimadores, profundidad máxima, muestras mínimas por división, etc.).

Utilizamos la clase `RandomizedSearchCV` configurada con 60 iteraciones. El algoritmo seleccionará 60 combinaciones al azar del diccionario. Para evaluar la capacidad de generalización de cada configuración, aplicamos validación cruzada (K-Fold = 3) utilizando F1-Macro como métrica de evaluación principal. 

Al finalizar, se extraerá el histórico completo de los resultados mediante el atributo `cv_results_` y se exportará en formato CSV para su posterior análisis.

In [2]:
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import RandomizedSearchCV
# from sklearn.ensemble import RandomForestClassifier

# # Definición del espacio de búsqueda para los hiperparámetros estructurales del Random Forest
# param_dist = {
#     'n_estimators': [100, 200, 300, 400, 500],
#     'max_depth': [None, 30, 50, 70, 100],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'max_features': ['sqrt', 'log2']
# }

# print("Espacio de hiperparámetros a evaluar:")
# for k, v in param_dist.items():
#     print(f" - {k}: {v}")

# # Instanciación del estimador base
# rf_base = RandomForestClassifier(random_state=42)

# # Configuración del objeto RandomizedSearchCV.
# # Se realizarán 60 muestreos aleatorios evaluados mediante validación cruzada (3 Folds).
# # La métrica objetivo se fija en 'f1_macro' para asegurar penalizaciones equitativas entre clases mayoritarias y minoritarias.
# random_search = RandomizedSearchCV(
#     estimator=rf_base,
#     param_distributions=param_dist,
#     n_iter=60,
#     scoring='f1_macro',
#     cv=3,
#     verbose=2,
#     random_state=42,
#     n_jobs=-1,
#     return_train_score=False
# )

# # Ejecución de la búsqueda iterativa sobre la matriz balanceada
# print("\nIniciando RandomizedSearchCV. Número de iteraciones: 60 | Validación Cruzada: 3 Folds.")
# random_search.fit(X_train_smote, y_train_smote_encoded)

# # Presentación de los resultados de convergencia
# print("\nOptimización finalizada.")
# print(f"Parámetros óptimos identificados: {random_search.best_params_}")
# print(f"F1-Macro máximo (K-Fold CV): {round(random_search.best_score_, 4)}")

# # Extracción de la memoria interna de validación cruzada (cv_results_) para su trazabilidad
# resultados_completos = pd.DataFrame(random_search.cv_results_)

# # Filtrado de variables de rendimiento y ordenación descendente según el ranking de test
# columnas_clave = ['rank_test_score', 'mean_test_score', 'std_test_score', 
#                   'param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
#                   'param_min_samples_leaf', 'param_max_features']

# df_exportar = resultados_completos[columnas_clave].sort_values(by='rank_test_score')

# # Estandarización de la nomenclatura de las columnas para el informe final
# df_exportar.rename(columns={
#     'rank_test_score': 'Ranking',
#     'mean_test_score': 'F1-Macro Medio',
#     'std_test_score': 'Desviacion Estandar'
# }, inplace=True)

# # Exportación del histórico iterativo
# ruta_exportacion = "../data/processed/historico_optimizacion_rf.csv"
# df_exportar.to_csv(ruta_exportacion, index=False)

# print(f"\nRegistro completo de arquitecturas exportado a: {ruta_exportacion}")
# display(df_exportar.head(5))

### Identificación de Anomalía Analítica: Fuga de Datos (Data Leakage)

La ejecución anterior arrojó un F1-Macro superior al 99%. Este resultado es metodológicamente inválido debido a un fenómeno conocido como Fuga de Datos (*Data Leakage*). 

Al aplicar SMOTE de manera global sobre todo el conjunto de entrenamiento antes de ejecutar la Validación Cruzada (K-Fold), los datos sintéticos clonados quedaron dispersos por todo el dataset. Como consecuencia, el algoritmo evaluó su rendimiento sobre datos sintéticos que el modelo ya había memorizado durante la fase de entrenamiento del pliegue (*fold*), produciendo una evaluación fraudulenta. Adicionalmente, la concurrencia masiva de subprocesos desencadenó errores de asignación de memoria RAM (`MemoryError`) en configuraciones algorítmicas de alta demanda computacional.

**Corrección Arquitectónica:**
Para garantizar la integridad matemática, se implementa una tubería dinámica (`Pipeline`) de la librería `imblearn`. Esta estructura asegura que la inyección sintética de SMOTE se ejecute exclusivamente sobre los fragmentos de entrenamiento de cada iteración de la validación cruzada, preservando la pureza de los fragmentos de evaluación. Paralelamente, se limitan los subprocesos del clasificador (`n_jobs=2`) para estabilizar el consumo de memoria.

In [ ]:
# from imblearn.pipeline import Pipeline as ImbPipeline
# from imblearn.over_sampling import SMOTE
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import RandomizedSearchCV
# import pandas as pd

# print("--- FASE 2.1: REEJECUCIÓN DE OPTIMIZACIÓN (PIPELINE DINÁMICO) ---\n")

# # 1. Construcción del Pipeline
# pipeline_rf = ImbPipeline([
#     # Bajamos k_neighbors a 2 para que sobreviva a los cortes del K-Fold
#     ('smote', SMOTE(random_state=42, k_neighbors=2)),
#     ('classifier', RandomForestClassifier(random_state=42, n_jobs=2)) 
# ])

# # 2. Redefinición del espacio de hiperparámetros
# # Los parámetros ahora deben llevar el prefijo 'classifier__' para que el Pipeline sepa a quién pertenecen
# param_dist_pipeline = {
#     'classifier__n_estimators': [100, 200, 300, 400, 500],
#     'classifier__max_depth': [None, 30, 50, 70, 100],
#     'classifier__min_samples_split': [2, 5, 10],
#     'classifier__min_samples_leaf': [1, 2, 4],
#     'classifier__max_features': ['sqrt', 'log2']
# }

# # 3. Configuración del Rastreador
# random_search_pipeline = RandomizedSearchCV(
#     estimator=pipeline_rf,
#     param_distributions=param_dist_pipeline,
#     n_iter=60,
#     scoring='f1_macro',
#     cv=3,
#     verbose=2,
#     random_state=42,
#     n_jobs=-1, # RandomizedSearch gestiona los workers generales
#     return_train_score=False
# )

# # 4. Ejecución sobre matriz pura (SIN SMOTE)
# print("Iniciando validación cruzada estructurada. Entrenando sobre matriz pura TF-IDF...")
# random_search_pipeline.fit(X_train_tfidf, y_train_encoded)

# # 5. Evaluación real y exportación
# print("\nOptimización finalizada con éxito.")
# print(f"Parámetros óptimos reales: {random_search_pipeline.best_params_}")
# print(f"F1-Macro máximo (K-Fold CV): {round(random_search_pipeline.best_score_, 4)}")

# # Generación del histórico corregido
# resultados_corregidos = pd.DataFrame(random_search_pipeline.cv_results_)

# columnas_clave_pipe = ['rank_test_score', 'mean_test_score', 'std_test_score', 
#                        'param_classifier__n_estimators', 'param_classifier__max_depth', 
#                        'param_classifier__min_samples_split', 'param_classifier__min_samples_leaf', 
#                        'param_classifier__max_features']

# df_exportar_corregido = resultados_corregidos[columnas_clave_pipe].sort_values(by='rank_test_score')

# # Estandarización de nomenclatura
# df_exportar_corregido.rename(columns={
#     'rank_test_score': 'Ranking',
#     'mean_test_score': 'F1-Macro Medio',
#     'std_test_score': 'Desviacion Estandar'
# }, inplace=True)

# # Exportación del histórico iterativo
# ruta_exportacion_corregida = "../data/processed/historico_optimizacion_rf_corregido.csv"
# df_exportar_corregido.to_csv(ruta_exportacion_corregida, index=False)

# print(f"\nRegistro auditado de arquitecturas exportado a: {ruta_exportacion_corregida}")
# display(df_exportar_corregido.head(5))

--- FASE 2.1: REEJECUCIÓN DE OPTIMIZACIÓN (PIPELINE DINÁMICO) ---

Iniciando validación cruzada estructurada. Entrenando sobre matriz pura TF-IDF...
Fitting 3 folds for each of 60 candidates, totalling 180 fits

Optimización finalizada con éxito.
Parámetros óptimos reales: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_features': 'log2', 'classifier__max_depth': 100}
F1-Macro máximo (K-Fold CV): 0.5838

Registro auditado de arquitecturas exportado a: ../data/processed/historico_optimizacion_rf_corregido.csv


,Ranking,F1-Macro Medio,Desviacion Estandar,param_classifier__n_estimators,param_classifier__max_depth,param_classifier__min_samples_split,param_classifier__min_samples_leaf,param_classifier__max_features
0,1,0.583818,0.014369,300,100,2,1,log2
23,2,0.567087,0.019665,500,None,5,1,sqrt
17,3,0.560299,0.013461,100,None,10,1,log2
55,4,0.547492,0.019887,200,None,10,1,sqrt
14,5,0.545940,0.020719,200,70,2,1,log2


### 2.2 Búsqueda Aleatoria Acotada (Smart Coarse Search)

La primera iteración demostró empíricamente que permitir al algoritmo explorar el espacio de hiperparámetros sin restricciones lógicas resulta contraproducente frente a matrices TF-IDF de alta dimensionalidad. La selección aleatoria de parámetros estructurales restrictivos (como `max_features='log2'`) limitó severamente la capacidad predictiva del modelo.

Se procede a ejecutar una segunda iteración de búsqueda aleatoria (`RandomizedSearchCV`), acotando el espacio matemático exclusivamente a rangos viables para el Procesamiento de Lenguaje Natural. El objetivo es identificar la zona óptima general (*Coarse Search*) para, posteriormente, someter a los parámetros ganadores a una búsqueda exhaustiva en cuadrícula (*Fine Search*) mediante `GridSearchCV`.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Definición del espacio de búsqueda acotado (54 combinaciones)
param_grid_coarse = {
    'classifier__n_estimators': [200, 400, 600],
    'classifier__max_features': ['sqrt', 0.05, 0.1],
    'classifier__max_depth': [None],
    'classifier__min_samples_split': [2, 5],
    'classifier__class_weight': ['balanced', 'balanced_subsample', None]
}

# 2. Configuración del buscador exhaustivo (GridSearchCV)
# Evaluará el 100% de las combinaciones posibles en este espacio.
grid_search_coarse = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_coarse,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,  # Paralelización máxima
    verbose=2
)

# 3. Ejecución del entrenamiento
print("Iniciando Fase 2.2: GridSearchCV exhaustivo...")
print("Iteraciones K-Fold: 3 | Combinaciones: 54 | Entrenamientos totales: 162")
grid_search_coarse.fit(X_train_tfidf, y_train_encoded)

print("\nBúsqueda Exhaustiva finalizada.")
print(f"Mejores parámetros encontrados: {grid_search_coarse.best_params_}")
print(f"F1-Macro máximo en validación (CV=3): {grid_search_coarse.best_score_:.4f}")

In [ ]:
import joblib

# Guardar el modelo ganador de esta fase (Coarse Search)
ruta_modelo_coarse = "../data/processed/sitor_rf_coarse_model.pkl"
joblib.dump(grid_search_coarse.best_estimator_, ruta_modelo_coarse)

print(f"Modelo de seguridad guardado en: {ruta_modelo_coarse}")